# Lezione 4 · Generalizzare e restare stabili

Il quaderno sottile, le notti rare, la bussola che mente, la serata assurda e la cima tra i bozzelli.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## I tre mucchi

*Nel racconto: «con il primo ti alleni… il terzo è l'esame: lo apri una volta sola».*

In [2]:
serate = 200
mucchi = [140, 30, 30]
assert sum(mucchi) == serate
for nome, n, quota in zip(["allenamento", "validazione", "test"], mucchi, [70, 15, 15]):
    controlla(f"{nome} (%)", n / serate * 100, quota)
k = 5
conta("validazione incrociata con k = 5: serate di verifica", serate // k, 40)
conta("validazione incrociata con k = 5: serate di allenamento", serate - serate // k, 160)

✓  allenamento (%): calcolato 70, nel libro 70
✓  validazione (%): calcolato 15, nel libro 15
✓  test (%): calcolato 15, nel libro 15
✓  validazione incrociata con k = 5: serate di verifica: calcolato 40, nel libro 40
✓  validazione incrociata con k = 5: serate di allenamento: calcolato 160, nel libro 160


## Overfitting, cioè i neuroni A e B

*Nel racconto: due neuroni perfetti sul quaderno sottile, opposti sulla sera nuova.*

In [3]:
quaderno = [((0.9, 0, 1), 1), ((0.7, 0, 1), 1), ((0.8, 0, 0), 1), ((0.3, 1, 0), 0), ((0.9, 0, 1), 1)]
A = (np.array([0, -3, 0]), 1.5)        # «è la pioggia a fermarti»
B = (np.array([6, 0, 0]), -3.0)        # «è la stanchezza a fermarti»
spinta = lambda neurone, x: float(neurone[0] @ np.array(x)) + neurone[1]

for nome, neurone in [("A", A), ("B", B)]:
    giuste = sum((spinta(neurone, x) > 0) == bool(y) for x, y in quaderno)
    conta(f"neurone {nome}: serate giuste su cinque", giuste, 5)
controlla("sera 4, neurone A", spinta(A, (0.3, 1, 0)), -1.5)
controlla("sera 4, neurone B", spinta(B, (0.3, 1, 0)), -1.2)
for (x, _), atteso in zip([quaderno[i] for i in (0, 1, 2, 4)], [2.4, 1.2, 1.8, 2.4]):
    controlla(f"neurone B, sera con energia {x[0]}", spinta(B, x), atteso)

nuova = (0.9, 1, 1)
controlla("sera nuova, neurone A: non vai", sigmoide(spinta(A, nuova)), 0.18)
controlla("sera nuova, neurone B: vai", sigmoide(spinta(B, nuova)), 0.92)
conta("generalization gap del primo kata (punti)", 99 - 70, 29)

✓  neurone A: serate giuste su cinque: calcolato 5, nel libro 5
✓  neurone B: serate giuste su cinque: calcolato 5, nel libro 5
✓  sera 4, neurone A: calcolato -1.5, nel libro -1.5
✓  sera 4, neurone B: calcolato -1.2, nel libro -1.2
✓  neurone B, sera con energia 0.9: calcolato 2.4, nel libro 2.4
✓  neurone B, sera con energia 0.7: calcolato 1.2, nel libro 1.2
✓  neurone B, sera con energia 0.8: calcolato 1.8, nel libro 1.8
✓  neurone B, sera con energia 0.9: calcolato 2.4, nel libro 2.4
✓  sera nuova, neurone A: non vai: calcolato 0.1824, nel libro 0.18
✓  sera nuova, neurone B: vai: calcolato 0.9168, nel libro 0.92
✓  generalization gap del primo kata (punti): calcolato 29, nel libro 29


## Quante serate bastano

*Nel racconto: «qualche decina di sere "no", non una sola».*

In [4]:
fatti = 3
conta("sere «no» con 10 casi per variabile", 10 * fatti, 30)
conta("sere «no» con 20 casi per variabile", 20 * fatti, 60)
conta("sere «no» nel quaderno sottile", sum(1 for _, y in quaderno if y == 0), 1)

✓  sere «no» con 10 casi per variabile: calcolato 30, nel libro 30
✓  sere «no» con 20 casi per variabile: calcolato 60, nel libro 60
✓  sere «no» nel quaderno sottile: calcolato 1, nel libro 1


## Le quattro discipline, per nome

*Nel racconto: il kata dall'altro lato, l'equipaggio incompleto, i terzaroli, smettere in tempo.*

In [5]:
eta, lam = 0.1, 0.01
controlla("i terzaroli: ogni peso moltiplicato per", 1 - eta * lam, 0.999, 0.0001)
controlla("dopo cento passi resta", (1 - eta * lam) ** 100, 0.905, 0.001)

# l'equipaggio incompleto: con p = 0,5 un neurone che vale 0,8 diventa 0 oppure 1,6
p = 0.5
valori = [0.0, 0.8 / (1 - p)]
controlla("dropout: il valore di chi resta acceso", valori[1], 1.6)
controlla("dropout: la media non cambia", p * valori[0] + (1 - p) * valori[1], 0.8)

✓  i terzaroli: ogni peso moltiplicato per: calcolato 0.999, nel libro 0.999
✓  dopo cento passi resta: calcolato 0.9048, nel libro 0.905
✓  dropout: il valore di chi resta acceso: calcolato 1.6, nel libro 1.6
✓  dropout: la media non cambia: calcolato 0.8, nel libro 0.8


## Le notti rare

*Nel racconto: diecimila notti di ormeggio, dieci ancore cedute, il modello pigro e quello utile.*

In [6]:
notti, cedute = 10_000, 10
VP, FN, FP = 8, 2, 40
VN = notti - VP - FN - FP
conta("veri negativi del modello utile", VN, 9950)
controlla("accuracy del modello utile (%)", (VP + VN) / notti * 100, 99.58)
controlla("recall del modello utile (%)", VP / (VP + FN) * 100, 80)
controlla("precision del modello utile (%)", VP / (VP + FP) * 100, 16.7, 0.05)
controlla("accuracy del modello pigro (%)", (notti - cedute) / notti * 100, 99.9)

# i costi del doppio ormeggio: mezz'ora per un falso allarme, cento ore per un'ancora persa
controlla("quaranta allarmi a vuoto (ore)", FP * 0.5, 20)
controlla("otto barche salvate (ore risparmiate)", VP * 100, 800)
controlla("il modello pigro costa (ore)", cedute * 100, 1000)
controlla("il modello utile costa (ore)", FN * 100 + FP * 0.5, 220)
assert 220 / 1000 < 1 / 4          # «meno di un quarto»

prec, rec = VP / (VP + FP), VP / (VP + FN)
controlla("F1 del modello utile", 2 * prec * rec / (prec + rec), 0.28)

✓  veri negativi del modello utile: calcolato 9.950, nel libro 9.950
✓  accuracy del modello utile (%): calcolato 99.58, nel libro 99.58
✓  recall del modello utile (%): calcolato 80, nel libro 80
✓  precision del modello utile (%): calcolato 16.67, nel libro 16.7
✓  accuracy del modello pigro (%): calcolato 99.9, nel libro 99.9
✓  quaranta allarmi a vuoto (ore): calcolato 20, nel libro 20
✓  otto barche salvate (ore risparmiate): calcolato 800, nel libro 800
✓  il modello pigro costa (ore): calcolato 1000, nel libro 1000
✓  il modello utile costa (ore): calcolato 220, nel libro 220
✓  F1 del modello utile: calcolato 0.2759, nel libro 0.28


## Errori pesati e probabilità da rimettere in scala

*Nel racconto: «un'ancora mancata può contare come cento notti tranquille sbagliate»; «prima di usarle, rimettile in scala».*

In [7]:
k = 100
controlla("un'ancora ceduta a cui il modello dava 0,1: punti persi", -math.log(0.1), 2.3)
controlla("   con k = 100", k * -math.log(0.1), 230, 0.5)

p = 0.5
quota_vera = (p / (1 - p)) / k
p_vera = quota_vera / (1 + quota_vera)
controlla("il modello allenato con k = 100 dice 50%: probabilità vera", p_vera, 1 / 101, 1e-9)
controlla("   cioè circa (%)", p_vera * 100, 1, 0.05)

✓  un'ancora ceduta a cui il modello dava 0,1: punti persi: calcolato 2.303, nel libro 2.3
✓     con k = 100: calcolato 230.3, nel libro 230
✓  il modello allenato con k = 100 dice 50%: probabilità vera: calcolato 0.009901, nel libro 0.00990099
✓     cioè circa (%): calcolato 0.9901, nel libro 1


## La deviazione, per nome

*Nel racconto: il custode che segna le serate «no» solo quando se ne ricorda; il martedì; i lupi sulla neve.*

In [8]:
si, no = 60, 40
no_segnati = no / 3
controlla("serate «no» che finiscono nel diario", no_segnati, 13, 0.5)
controlla("il diario dice che vai (%)", si / (si + round(no_segnati)) * 100, 82, 0.5)
controlla("la verità (%)", si / (si + no) * 100, 60)

✓  serate «no» che finiscono nel diario: calcolato 13.33, nel libro 13
✓  il diario dice che vai (%): calcolato 82.19, nel libro 82
✓  la verità (%): calcolato 60, nel libro 60


## Più bussole: l'ensemble

*Nel racconto: «porta più bussole, montate su barche diverse».*

In [9]:
d_accordo = [0.90, 0.88, 0.91, 0.89, 0.92]
in_disaccordo = [0.95, 0.20, 0.70, 0.10, 0.55]
controlla("cinque bussole d'accordo: media", np.mean(d_accordo), 0.90)
controlla("cinque bussole d'accordo: deviazione standard, «poco più di 0,01»", np.std(d_accordo), 0.014, 0.001)
controlla("cinque bussole in disaccordo: media", np.mean(in_disaccordo), 0.50)
controlla("cinque bussole in disaccordo: deviazione standard", np.std(in_disaccordo), 0.31)

✓  cinque bussole d'accordo: media: calcolato 0.9, nel libro 0.9
✓  cinque bussole d'accordo: deviazione standard, «poco più di 0,01»: calcolato 0.01414, nel libro 0.014
✓  cinque bussole in disaccordo: media: calcolato 0.5, nel libro 0.5
✓  cinque bussole in disaccordo: deviazione standard: calcolato 0.3146, nel libro 0.31


## Fuori distribuzione e deriva

*Nel racconto: l'energia 5; «tira righe alla cieca»; il dojo che si trasferisce al coperto.*

In [10]:
w, b = np.array([2.0, -1.5, 0.1]), -0.5
z = float(w @ [5, 1, 1]) + b
controlla("la spinta della serata assurda", z, 8.1)
controlla("la risposta del neurone", sigmoide(z), 0.9997, 0.0001)
# più la serata è assurda, più il modello è sicuro
assert sigmoide(float(w @ [50, 1, 1]) + b) > sigmoide(z)

✓  la spinta della serata assurda: calcolato 8.1, nel libro 8.1
✓  la risposta del neurone: calcolato 0.9997, nel libro 0.9997


## La cima tra i bozzelli: gradienti che svaniscono ed esplodono

*Nel racconto: la cima che passa per cinquanta bozzelli; il paranco.*

In [11]:
controlla("un quarto, per dieci strati (milionesimi)", 0.25 ** 10 * 1e6, 0.95)
controlla("fattori di 1,5, per cinquanta strati (centinaia di milioni)", 1.5 ** 50 / 1e8, 6.4, 0.05)

# gradient clipping con soglia 1
g = np.array([3.0, 4.0])
controlla("lunghezza del gradiente (3; 4)", np.linalg.norm(g), 5)
accorciato = g / np.linalg.norm(g)
controlla("accorciato: prima componente", accorciato[0], 0.6)
controlla("accorciato: seconda componente", accorciato[1], 0.8)

✓  un quarto, per dieci strati (milionesimi): calcolato 0.9537, nel libro 0.95
✓  fattori di 1,5, per cinquanta strati (centinaia di milioni): calcolato 6.376, nel libro 6.4
✓  lunghezza del gradiente (3; 4): calcolato 5, nel libro 5
✓  accorciato: prima componente: calcolato 0.6, nel libro 0.6
✓  accorciato: seconda componente: calcolato 0.8, nel libro 0.8


## Partire con la forza giusta: l'inizializzazione

*Nel racconto: «i pesi di partenza si scelgono della misura giusta».*

In [12]:
controlla("He con 256 ingressi: deviazione standard dei pesi", math.sqrt(2 / 256), 0.088, 0.0005)

✓  He con 256 ingressi: deviazione standard dei pesi: calcolato 0.08839, nel libro 0.088


## Rimettere in misura: la normalizzazione

*Nel racconto: il marinaio a ogni bozzello che ritende o lasca la cima.*

In [13]:
x = np.array([2.0, 4.0, 6.0])
controlla("media", x.mean(), 4)
controlla("varianza", x.var(), 2.67)
controlla("deviazione standard", x.std(), 1.63)
for v, a in zip((x - x.mean()) / x.std(), [-1.22, 0, 1.22]):
    controlla("LayerNorm", v, a)
rms = math.sqrt(np.mean(x ** 2))
controlla("radice della media dei quadrati", rms, 4.32)
for v, a in zip(x / rms, [0.46, 0.93, 1.39]):
    controlla("RMSNorm", v, a)

✓  media: calcolato 4, nel libro 4
✓  varianza: calcolato 2.667, nel libro 2.67
✓  deviazione standard: calcolato 1.633, nel libro 1.63
✓  LayerNorm: calcolato -1.225, nel libro -1.22
✓  LayerNorm: calcolato 0, nel libro 0
✓  LayerNorm: calcolato 1.225, nel libro 1.22
✓  radice della media dei quadrati: calcolato 4.32, nel libro 4.32
✓  RMSNorm: calcolato 0.4629, nel libro 0.46
✓  RMSNorm: calcolato 0.9258, nel libro 0.93
✓  RMSNorm: calcolato 1.389, nel libro 1.39


## La via diretta: le connessioni residue

*Nel racconto: la seconda cima tesa diritta dalla mano alla vela.*

In [14]:
a = b_ = 0.01
controlla("senza vie dirette risale", a * b_, 0.0001, 1e-9)
controlla("con le vie dirette risale", (1 + a) * (1 + b_), 1.0201, 1e-9)

✓  senza vie dirette risale: calcolato 0.0001, nel libro 0.0001
✓  con le vie dirette risale: calcolato 1.02, nel libro 1.0201


## Kata per casa

**Esercizio 2.** La regola di controllo che avrebbe fermato la serata assurda, scritta in codice.

In [15]:
def controlla_serata(energia, pioggia, compagno):
    """Che cosa controlla, che cosa fa se il controllo fallisce, chi viene avvisato."""
    if not (isinstance(energia, (int, float)) and 0 <= energia <= 1):
        return "non chiamo il modello: l'energia deve stare tra 0 e 1 → registro l'anomalia, decide una regola prudente o una persona"
    if pioggia not in (0, 1) or compagno not in (0, 1):
        return "non chiamo il modello: pioggia e compagno devono valere 0 o 1 → registro l'anomalia, decide una regola prudente o una persona"
    return "ok, chiamo il modello"

for serata in [(0.8, 1, 1), (5, 1, 1), (0.8, 2, 1)]:
    print(serata, "→", controlla_serata(*serata))
assert controlla_serata(0.8, 1, 1).startswith("ok") and not controlla_serata(5, 1, 1).startswith("ok")

(0.8, 1, 1) → ok, chiamo il modello
(5, 1, 1) → non chiamo il modello: l'energia deve stare tra 0 e 1 → registro l'anomalia, decide una regola prudente o una persona
(0.8, 2, 1) → non chiamo il modello: pioggia e compagno devono valere 0 o 1 → registro l'anomalia, decide una regola prudente o una persona
